# 1.10 影響怎麼穿過多個運算？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：旋鈕往哪邊轉，猜錯的代價會下降？**

藍線是 (w−3)²；橘點到綠點只走一小步。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/gradient.svg")))

**先想一想：**兩條變化率相乘，為什麼不是相加？

把整條計算看成兩個相連的局部變化：w 改一點，u 改多少；u 改一點，loss 改多少。乘起來就知道最終敏感度。

**把直覺寫成數學：**$u=2w$，$L=u^2$；$dL/dw=(dL/du)(du/dw)=2u\cdot2$。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
w = torch.tensor(3.0, requires_grad=True)
u = 2 * w
loss = u**2
loss.backward()
print("局部", 2 * u.detach(), 2, "合成", w.grad)
assert w.grad.item() == 24

**如何讀結果：**鏈式法則是 autograd 的基礎；backward 遍歷已建立的計算圖，不猜公式。

**只改一件事：**只把 u=2w 改為 u=3w，手算再核對。
